In [1]:
pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [5]:
def scrape_coinmarketcap(total_pages):
    all_coins_data = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    }

    for page in range(1, total_pages + 1):
        print(f"Scrapping Page: {page}...")
        url = f"https://coinmarketcap.com/?page={page}"
        
        try:
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Table find karna
            table = soup.find('table', class_='cmc-table')
            if not table:
                print(f"Page {page} par table nahi mila.")
                continue
                
            rows = table.find('tbody').find_all('tr')

            for index, row in enumerate(rows):
                # Condition: 1st page ka 1st row skip karna hai
                if page == 1 and index == 0:
                    continue
                
                cells = row.find_all('td')
                
                # Check agar row mein data hai (kabhi kabhi ads row aa jati hain)
                if len(cells) < 10:
                    continue

                # Data Extraction
                try:
                    data = {
                        "Rank": cells[1].get_text(strip=True),
                        "Name": cells[2].find('p', class_='coin-item-name').get_text(strip=True) if cells[2].find('p', class_='coin-item-name') else cells[2].find('span', class_='crypto-symbol').get_text(strip=True),
                        "Price": cells[3].get_text(strip=True),
                        "1h%": cells[4].get_text(strip=True),
                        "24h%": cells[5].get_text(strip=True),
                        "7d%": cells[6].get_text(strip=True),
                        "Market Cap": cells[7].find('span', class_='sc-11478e5d-0').get_text(strip=True) if cells[7].find('span') else cells[7].get_text(strip=True),
                        "Volume(24h)": cells[8].find('p').get_text(strip=True) if cells[8].find('p') else cells[8].get_text(strip=True)
                    }
                    all_coins_data.append(data)
                except AttributeError:
                    continue

            # Har page ke baad thoda gap dena behtar hai taake block na hon
            time.sleep(2)

        except Exception as e:
            print(f"Error on page {page}: {e}")
            break

    # Pandas DataFrame mein convert karna
    df = pd.DataFrame(all_coins_data)
    return df

In [6]:
df_crypto = scrape_coinmarketcap(total_pages=2)

Scrapping Page: 1...
Scrapping Page: 2...


In [7]:
df_crypto

,Rank,Name,Price,1h%,24h%,7d%,Market Cap,Volume(24h)
0,1,Bitcoin,"$80,074.37",1.48%,1.80%,2.36%,$1.59T,"$45,094,955,140"
1,2,Ethereum,"$2,367.26",1.11%,1.88%,1.67%,$284.02B,"$22,402,210,819"
2,3,Tether,$0.9997,0.01%,0.00%,0.05%,$189.53B,"$138,793,308,342"
3,4,XRP,$1.40,0.37%,0.57%,0.47%,$86.63B,"$2,231,367,497"
4,5,BNB,$626.12,0.35%,1.07%,0.35%,$84.34B,"$2,107,281,764"
5,6,USDC,$0.9997,0.00%,0.00%,0.01%,$77.49B,"$61,433,555,635"
6,7,Solana,$84.76,0.75%,0.54%,0.97%,$48.85B,"$4,776,987,007"
7,8,TRON,$0.3393,0.53%,0.52%,4.04%,$32.17B,"$800,736,246"
8,9,Dogecoin,$0.1116,0.64%,2.84%,12.68%,$18.97B,"$2,341,787,142"
9,10,Hyperliquid,$41.29,0.53%,0.45%,3.68%,$10.52B,"$286,916,951"


In [8]:
df_crypto.to_csv('coinmarketcap_data_bs4.csv', index=False)